### Inferring
- In this lesson, you will infer sentiment and topics from product reviews and news articles.

### Setup

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from huggingface_hub import InferenceClient

# Load environment variables
#_ = load_dotenv(find_dotenv())

# Get Hugging Face API token
API_token = os.getenv("my api")

# Create client
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token= API_token
)

In [2]:
# helper function
def get_completion(prompt):
    response = client.chat_completion(
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

### Product review test

In [3]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

### Sentiment (positive/negative)

In [4]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the product review is overwhelmingly positive. 

The reviewer mentions several positive aspects of the product and the company:

1. The lamp met their expectations in terms of price and features.
2. The company provided excellent customer service, replacing a broken part quickly and efficiently.
3. The support team was responsive and helpful in resolving an issue with a missing part.
4. The reviewer feels that the company genuinely cares about its customers and products.

There are no negative comments or criticisms in the review, which further emphasizes the positive sentiment. The reviewer's enthusiasm and appreciation for the company's customer service and products are evident throughout the review.


In [5]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

positive


### Identify types of emotions

In [6]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

gratitude, satisfaction, relief, trust, appreciation


### Identify anger

In [7]:
prompt = f"""
Is the writer of the following review expressing anger?
The review is delimited with triple backticks. 
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No. 

The writer of the review is expressing satisfaction and appreciation for the company's customer service and product quality, rather than anger.


### Extract product and company name from customer reviews

In [8]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

```json
{
  "Item": "lamp",
  "Brand": "Lumina"
}
```


### Doing multiple tasks at once

In [9]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

```json
{
  "Sentiment": "positive",
  "Anger": false,
  "Item": "lamp",
  "Brand": "Lumina"
}
```


### Inferring topics

In [10]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

### Infer 5 topics

In [11]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

NASA satisfaction, Government survey, Job ratings, Employee morale, Department rankings


In [12]:
response.split(sep=',')

['NASA satisfaction',
 ' Government survey',
 ' Job ratings',
 ' Employee morale',
 ' Department rankings']

### Make a news alert for certain topics

In [18]:
prompt = f"""
Determine whether each topic is present in the text.

Return ONLY a valid JSON object.
Use 1 if the topic is present and 0 otherwise.

Topics:
{topic_list}

Text:
'''{story}'''
"""

In [19]:
import json

response = get_completion(prompt)

topic_dict = json.loads(response)

if topic_dict.get("nasa") == 1:
    print("ALERT: New NASA story!")

ALERT: New NASA story!
